# 切口 A · Step 2 — 用户活跃度演变与跨层迁移分析(从假设到反常洞察)

**这是切口 A 的"反常发现补充"**。

## 缘起

在 04 notebook 看 D3-7 衰减分布时,我们发现了一个反常现象:
- 低活跃用户 D3-7 日均反而是 D1-2 的 3.4 倍(衰减 -243%)
- 中活跃用户 D3-7 比 D1-2 略涨(衰减 -25%)
- 只有高活跃用户呈现真正的下降(衰减 ~40%)

**真正"近假死"(D3-7 = 0)的用户只有 ~1%,远低于业界经验**。

这个反常现象的统计学解释是 **回归均值(Regression to the Mean)**:
基于短窗口(2 天)切的 tier,本身就有大量随机噪声,后续观察会自然"回归"。

## 本 notebook 干什么

1. 重新审视跨层迁移现象:谁跃升、谁维持、谁跌落?
2. 对三组迁移用户做 Day 1-2 行为特征对比
3. 把"回归均值"作为方法论自省,落到 README 里
4. 产出业务建议:**不能用 Day 1-2 单窗口预测后续 tier**,真正有预测力的是哪些特征?

**这部分内容在简历和面试里的价值**:
- 一般作品只讲"我做了 X 分析,得到 Y 结论"
- 我们多一层:"我假设是 Z,数据告诉我不是,深挖发现是回归均值现象"
- 这种"假设 → 反常 → 重新解释"的叙事,是分析师素养的直接证据

## 0. 环境与数据

In [1]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

PARQUET_PATH = Path('../data/processed/DWD/user_behavior_clean.parquet').resolve()
con = duckdb.connect()
con.execute(f"""
CREATE OR REPLACE VIEW ub AS
SELECT user_id, item_id, category_id, behavior_type, timestamp,
       CAST(to_timestamp(timestamp) AS DATE) AS event_date
FROM '{PARQUET_PATH.as_posix()}'
""")
print('视图 ub 已建立')

视图 ub 已建立


---

## 1. 重建 cohort + D1-2 tier + D3-7 tier

In [ ]:
cohort_df = con.sql("""
WITH cohort AS (
    SELECT DISTINCT user_id FROM ub
    WHERE event_date IN (DATE '2017-11-25', DATE '2017-11-26')
),
d12_features AS (
    SELECT
        user_id,
        COUNT(*)                                                AS d12_actions,
        SUM(CASE WHEN behavior_type='pv'   THEN 1 ELSE 0 END)   AS d12_pv,
        SUM(CASE WHEN behavior_type='cart' THEN 1 ELSE 0 END)   AS d12_cart,
        SUM(CASE WHEN behavior_type='fav'  THEN 1 ELSE 0 END)   AS d12_fav,
        SUM(CASE WHEN behavior_type='buy'  THEN 1 ELSE 0 END)   AS d12_buy,
        COUNT(DISTINCT item_id)                                 AS d12_unique_items,
        COUNT(DISTINCT category_id)                             AS d12_unique_categories
    FROM ub
    WHERE event_date IN (DATE '2017-11-25', DATE '2017-11-26')
    GROUP BY user_id
),
d37 AS (
    SELECT user_id, COUNT(*) AS d37_actions
    FROM ub
    WHERE event_date BETWEEN DATE '2017-11-27' AND DATE '2017-12-01'
    GROUP BY user_id
)
SELECT
    c.user_id,
    f.d12_actions, f.d12_pv, f.d12_cart, f.d12_fav, f.d12_buy,
    f.d12_unique_items, f.d12_unique_categories,
    COALESCE(d.d37_actions, 0)         AS d37_actions,
    COALESCE(d.d37_actions, 0) / 5.0   AS d37_per_day
FROM cohort c
JOIN d12_features f ON c.user_id = f.user_id
LEFT JOIN d37 d ON c.user_id = d.user_id
""").df()

# D1-2 tier
cohort_df['d12_tier'] = pd.cut(cohort_df['d12_actions'],
                               bins=[0, 8, 25, float('inf')],
                               labels=['低活跃', '中活跃', '高活跃'])

# D3-7 tier(用日均阈值,加"近假死"层)
def d37_tier(per_day):
    if per_day < 0.2:        return '近假死'
    elif per_day <= 4:       return '低活跃'
    elif per_day <= 12.5:    return '中活跃'
    else:                    return '高活跃'

cohort_df['d37_tier'] = cohort_df['d37_per_day'].apply(d37_tier)

# 迁移方向(数值化对比)
# 注意:pd.cut 返回 Categorical,.map() 后仍是分类型,不能直接相减
# 用 .astype('int64') 强制转成数值
tier_to_num_d12 = {'低活跃': 1, '中活跃': 2, '高活跃': 3}
tier_to_num_d37 = {'近假死': 0, '低活跃': 1, '中活跃': 2, '高活跃': 3}

cohort_df['d12_num'] = cohort_df['d12_tier'].map(tier_to_num_d12).astype('int64')
cohort_df['d37_num'] = cohort_df['d37_tier'].map(tier_to_num_d37).astype('int64')
cohort_df['migration_delta'] = cohort_df['d37_num'] - cohort_df['d12_num']

def migration_label(delta):
    if delta > 0:   return '跃升'
    elif delta < 0: return '跌落'
    else:           return '维持'

cohort_df['migration'] = cohort_df['migration_delta'].apply(migration_label)

print(f'cohort 总用户: {len(cohort_df):,}')
print()
print('三种迁移人群占比:')
print(cohort_df['migration'].value_counts(normalize=True).map(lambda x: f'{x*100:.2f}%'))

---

## 2. 跨层迁移热力图(从 04 notebook 复用,放在这做主分析锚点)

In [ ]:
trans = pd.crosstab(cohort_df['d12_tier'], cohort_df['d37_tier'], normalize='index') * 100
col_order = ['近假死', '低活跃', '中活跃', '高活跃']
trans = trans[col_order]

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(trans, annot=True, fmt='.1f', cmap='YlGnBu',
            cbar_kws={'label': '% (按行归一化)'}, linewidths=0.5, ax=ax)
ax.set_title('跨层迁移矩阵:Day 1-2 tier → Day 3-7 tier')
ax.set_xlabel('Day 3-7 tier')
ax.set_ylabel('Day 1-2 tier')
plt.tight_layout()
plt.show()

print()
print('每层迁移方向分解:')
mig_breakdown = cohort_df.groupby(['d12_tier', 'migration'], observed=True).size().unstack(fill_value=0)
mig_breakdown_pct = mig_breakdown.div(mig_breakdown.sum(axis=1), axis=0) * 100
print(mig_breakdown_pct.round(2))

**这张图的核心读法**:
- **对角线(深色)= 维持原 tier**
- 对角线右上方(偏向"高活跃")= 跃升
- 对角线左下方(偏向"近假死")= 跌落
- 注意每**行**加起来 = 100%,所以读"低活跃行"就是"low tier users 后来去哪了"

回归均值导致:
- 低活跃 → 大量跃升(50%+)
- 高活跃 → 大量跌落(45%+)
- 中活跃 → 两端都有

---

## 3. 三组迁移用户的 Day 1-2 特征对比

**重点**:对每个 D1-2 tier 内部,比较跃升 / 维持 / 跌落三组的 Day 1-2 行为差异。
找出"是哪些特征能预测未来 tier 走向"。

In [ ]:
features = ['d12_actions', 'd12_pv', 'd12_cart', 'd12_fav', 'd12_buy',
            'd12_unique_items', 'd12_unique_categories']

# 给每个 tier 算三组(跃升/维持/跌落)的特征均值
result = cohort_df.groupby(['d12_tier', 'migration'], observed=True)[features].mean().round(2)
print('每层 × 迁移方向 的 Day 1-2 特征均值:')
print(result)

**这张表的读法**(以低活跃为例):
- 看"低活跃-跃升"那行 vs "低活跃-维持"那行
- 哪些特征"跃升组"明显更高?(类目宽度?加购?)
- **如果跃升组在 Day 1-2 就有更多 cart/fav/buy 行为,说明意向行为是跃升的早期信号**

同理:
- "高活跃-跌落"组 vs "高活跃-维持"组,哪些特征预示跌落?
- 如果跌落组 Day 1-2 浏览类目特别窄,可能是"一次性来逛一次"用户

## 4. 可视化:每层各特征的"跃升 vs 跌落"对比

In [ ]:
# 只画几个关键特征
key_features = ['d12_actions', 'd12_unique_categories', 'd12_cart', 'd12_buy']
feature_labels = {'d12_actions': '总行为数', 'd12_unique_categories': '类目宽度',
                  'd12_cart': '加购次数', 'd12_buy': '购买次数'}

fig, axes = plt.subplots(1, len(key_features), figsize=(18, 4.5))

for ax, feat in zip(axes, key_features):
    pivot = cohort_df.pivot_table(values=feat, index='d12_tier',
                                   columns='migration', aggfunc='mean', observed=True)
    # 列顺序保证一致
    pivot = pivot[['跃升', '维持', '跌落']] if all(c in pivot.columns for c in ['跃升', '维持', '跌落']) else pivot
    pivot.plot(kind='bar', ax=ax, color=['#5BA8A8', '#A8A8A8', '#E07B7B'])
    ax.set_title(f'{feature_labels[feat]}')
    ax.set_xlabel('D1-2 tier')
    ax.set_ylabel('均值')
    ax.set_xticklabels(pivot.index, rotation=0)
    ax.legend(title='迁移方向', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Day 1-2 特征:跃升 vs 维持 vs 跌落(分层对比)', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

**关键观察点**:
- **在同一个 tier 内,跃升组和跌落组在 Day 1-2 是否真的有差异?**
- 如果差异小 → 印证回归均值(差异只是随机的)
- 如果差异大 → 说明这些特征有预测能力

业务含义:
- 差异大 → 可以用 Day 1-2 特征做用户分流预测
- 差异小 → 不能光靠首日特征预判,需要更长观察期

---

## 5. 回归均值的方法论自省

这一节是 markdown only,直接写进 README 的"分析师自省"章节。

### 5.1 什么是回归均值

> 在统计学中,**回归均值**指:对一个总体进行测量,选取测量值极端的子样本,
> 再次测量该子样本时,其平均值往往会向总体均值回归。

### 5.2 为什么 cohort 分析特别容易遇到

把用户按某次测量分 tier 时,**测量本身有噪声**:
- 同一个"真实活跃水平 = 5次/天"的用户,在不同 2 天窗口测出来可能是 3、4、6、8
- 我们抽到 "Day 1-2 = 3" 的样本时,这个人本质上 5 次/天,后续观察自然会"回升"
- 反过来,Day 1-2 = 8 的样本,后续会"回落"

### 5.3 我们这个项目的具体表现

在 9 天数据 + 2 天分类窗口的条件下,回归均值效应被放大:
- 低活跃 → 大量看起来"跃升"
- 高活跃 → 大量看起来"跌落"
- 实质上很大一部分人就是回到了自己的真实水平

### 5.4 如何缓解(理论上)

| 方法 | 我们能否实施 |
|---|---|
| 用更长的分类窗口(如 7 天) | ❌ 数据只有 9 天,做不到 |
| 用多窗口稳健平均代替单窗口测量 | ⚠️ 部分可行,但削弱后续观察期 |
| 引入更稳健的"用户类型"指标(不只是行为数) | ✅ 可以加入类目宽度、转化深度等 |
| 接受现实并诚实声明 caveat | ✅ **本项目采取的路线** |

### 5.5 给业务的提醒

**用单次窗口测量预测用户未来 tier,是有偏的**。生产环境中:
- 用"近 N 天滚动平均"代替"昨天的一次测量"
- 对"突然活跃度异常"的用户先观察再行动
- 在 A/B 实验里特别留意这种"看似改变"实则"自然回归"的现象

---

## 6. 业务建议(等你填)

> 跑完上面所有 cell 后,根据 Part 3 / Part 4 的对比结果,填以下决策

### 6.1 跃升预警信号
> 在低活跃用户里,Day 1-2 满足什么特征的人,后续跃升概率明显更高?
> 这些人是"被低估的优质用户",值得优先曝光
> 关键特征 1:___
> 关键特征 2:___

### 6.2 跌落预警信号
> 在高活跃用户里,Day 1-2 满足什么特征的人,后续跌落概率明显更高?
> 这些人是"昙花一现的用户",不值得过早投入推广预算
> 关键特征 1:___
> 关键特征 2:___

### 6.3 总结:这个项目要怎么讲故事
> 一句话概括从假设到反常洞察的完整故事:
> ___

### 接下来
- 与 05 notebook 的"假死画像"结合写 README 的切口 A 章节
- 然后进入切口 C(探索 vs 收割用户分群)
